# 11: Capstone End-to-End Workflow

This capstone solves one synthetic classification problem by running a full pySurgery pipeline:
1. build discrete models,
2. extract `pi_1` evidence,
3. compute Whitehead/Wall/structure-set data,
4. run dimension-aware homeomorphism analyzers,
5. generate witness objects,
6. produce a surgery-level plan when an obstruction appears.


## Problem Setup

Assume we are studying two candidate spaces `A` and `B` from a geometric pipeline.
We want classification diagnostics across dimensions and one final high-dimensional decision with certificates.

In [ ]:
import numpy as np
import scipy.sparse as sp

from pysurgery import (
    ChainComplex,
    CWComplex,
    FundamentalGroup,
    IntersectionForm,
    ObstructionResult,
    ProductAssemblyCertificate,
    HomotopyCompletionCertificate,
    ThreeManifoldRecognitionCertificate,
    DefiniteLatticeIsometryCertificate,
    WhiteheadGroup,
    StructureSet,
    build_homeomorphism_witness,
    compute_whitehead_group,
    evaluate_phase2_readiness,
    extract_pi_1,
    extract_pi_1_with_traces,
    compute_optimal_h1_basis_from_simplices,
    analyze_homeomorphism_2d_result,
    analyze_homeomorphism_3d_result,
    analyze_homeomorphism_4d_result,
    analyze_homeomorphism_high_dim_result,
    surgery_to_remove_impediments,
)


## Stage 1: Fundamental Group Extraction and Readiness


In [ ]:
cw = CWComplex(cells={0: 1, 1: 1}, attaching_maps={1: sp.csr_matrix((1, 1))})
pi_from_data = extract_pi_1(cw)
print('extracted pi_1:', pi_from_data)

pi_with_traces = extract_pi_1_with_traces(cw)
print('trace generators:', pi_with_traces.generators)
for tr in pi_with_traces.traces:
    print('  ', tr.generator, 'edge_index=', tr.edge_index, 'directed_path=', tr.directed_edge_path)

wh_from_data = compute_whitehead_group(pi_from_data)
print('Whitehead summary:', wh_from_data.description)

phase2 = evaluate_phase2_readiness(pi_from_data, 'Z')
print('phase2 readiness:', phase2.model_dump())

# Compute H_1 generators as data-native edge supports
simplices_h1 = [(0, 1), (1, 2), (2, 3), (0, 3)]
h1_basis = compute_optimal_h1_basis_from_simplices(simplices_h1, num_vertices=4)
print('H_1 rank:', h1_basis.rank)
for i, g in enumerate(h1_basis.generators):
    print(f'  generator[{i}] edges =', g.support_edges)


## Stage 2: Dimension-Aware Quick Diagnostics


In [ ]:
# 2D toy sphere-like complexes
c2a = ChainComplex(boundaries={1: sp.csr_matrix((1, 0)), 2: sp.csr_matrix((0, 1))}, dimensions=[0, 1, 2])
c2b = ChainComplex(boundaries={1: sp.csr_matrix((1, 0)), 2: sp.csr_matrix((0, 1))}, dimensions=[0, 1, 2])
r2 = analyze_homeomorphism_2d_result(c2a, c2b)
print('2D:', r2.status, '-', r2.reasoning[:90])

# 3D lens-like toy + decision-ready recognition certificate
d1 = sp.csr_matrix(np.zeros((1, 1), dtype=np.int64))
d2 = sp.csr_matrix(np.array([[3]], dtype=np.int64))
d3 = sp.csr_matrix(np.zeros((1, 1), dtype=np.int64))
c3a = ChainComplex(boundaries={1: d1, 2: d2, 3: d3}, dimensions=[0, 1, 2, 3], cells={0: 1, 1: 1, 2: 1, 3: 1})
c3b = ChainComplex(boundaries={1: d1, 2: d2, 3: d3}, dimensions=[0, 1, 2, 3], cells={0: 1, 1: 1, 2: 1, 3: 1})
pi3 = FundamentalGroup(generators=['g'], relations=[['g', 'g', 'g']])
rec3 = ThreeManifoldRecognitionCertificate(provided=True, source='capstone', exact=True, validated=True)
r3 = analyze_homeomorphism_3d_result(c3a, c3b, pi1_1=pi3, pi1_2=pi3, recognition_certificate=rec3)
print('3D:', r3.status, '-', r3.reasoning[:90])


In [4]:
# 4D definite case with explicit certificate
Q1 = np.array([[1, 0], [0, 1]], dtype=np.int64)
U = np.array([[3, 2], [2, 1]], dtype=np.int64)
Q2 = U.T @ Q1 @ U
f4a = IntersectionForm(matrix=Q1, dimension=4)
f4b = IntersectionForm(matrix=Q2, dimension=4)
cert4 = DefiniteLatticeIsometryCertificate(
    provided=True, source='capstone', exact=True, validated=True, isometry_matrix=U.tolist()
)
r4 = analyze_homeomorphism_4d_result(
    f4a, f4b, ks1=0, ks2=0, simply_connected=True,
    definite_lattice_isometry_certificate=cert4,
)
print('4D:', r4.status, '-', r4.reasoning[:90])


4D: success - SUCCESS: Definite lattice isomorphism certificate found (U^T Q1 U = Q2).


## Stage 3: High-Dimensional End-to-End Classification

We run one exact success path (with certificates) and one obstruction path (`surgery_required`).

In [ ]:
d1h = sp.csr_matrix(np.zeros((1, 0), dtype=np.int64))
d5h = sp.csr_matrix(np.zeros((0, 1), dtype=np.int64))
c5a = ChainComplex(boundaries={1: d1h, 5: d5h}, dimensions=[0, 1, 2, 3, 4, 5], cells={0: 1, 1: 0, 2: 0, 3: 0, 4: 0, 5: 1})
c5b = ChainComplex(boundaries={1: d1h, 5: d5h}, dimensions=[0, 1, 2, 3, 4, 5], cells={0: 1, 1: 0, 2: 0, 3: 0, 4: 0, 5: 1})

wh_ok = WhiteheadGroup(rank=0, description='Wh(Z_2 x Z_3)=0', computable=True, exact=True)
wall_ok = ObstructionResult(
    dimension=5, pi='Z_2 x Z_3', computable=True, exact=True, value=0, modulus=None, assumptions=[],
    decomposition_kind='factor_surrogate', assembly_certified=False, obstructs=False, zero_certified=True,
)
completion = HomotopyCompletionCertificate(provided=True, source='capstone', exact=True, validated=True)
assembly = ProductAssemblyCertificate(provided=True, source='capstone', exact=True, validated=True)
r5_success = analyze_homeomorphism_high_dim_result(
    c5a, c5b, dim=5, pi_group='Z_2 x Z_3', whitehead_group=wh_ok, wall_obstruction=wall_ok,
    homotopy_completion_certificate=completion, product_assembly_certificate=assembly,
)
print('5D success path:', r5_success.status, '-', r5_success.reasoning[:90])

wall_bad = ObstructionResult(dimension=5, pi='1', computable=True, exact=True, value=1, modulus=None, assumptions=[])
r5_obstructed = analyze_homeomorphism_high_dim_result(
    c5a, c5b, dim=5, pi1=FundamentalGroup(generators=[], relations=[]),
    whitehead_group=WhiteheadGroup(rank=0, description='Wh(1)=0', computable=True, exact=True),
    wall_obstruction=wall_bad,
)
print('5D obstruction path:', r5_obstructed.status, '-', r5_obstructed.reasoning[:90])


## Stage 4: Witness Construction and Audit Payloads


In [ ]:
w = build_homeomorphism_witness(
    c1=c5a, c2=c5b, dim=5,
    pi_group='Z_2 x Z_3',
    whitehead_group=wh_ok,
    wall_obstruction=wall_ok,
    homotopy_completion_certificate=completion,
    product_assembly_certificate=assembly,
)
print('witness status:', w.status)
if w.witness is not None:
    print('witness kind:', w.witness.kind)
    print('certificate keys sample:', sorted(list(w.witness.certificates.keys()))[:8])


## Stage 5: Structure Set and Surgery Planning


In [ ]:
sset = StructureSet(dimension=5, fundamental_group='1')
seq = sset.evaluate_exact_sequence_result()
print('sequence computable/exact:', seq.computable, seq.exact)

can_remove, plan = surgery_to_remove_impediments(f4a, target_sig=5)
print('surgery planning possible:', can_remove)
print(plan)


## Final Notes

This is one end-to-end workflow. In real pipelines, replace toy chain complexes and certificates with data-derived objects from your TDA/geometry stack, then keep the same decision structure.